# 05 — Geographic Data Preparation

## 1. Introduction

### Objective

This notebook prepares the dataset for **Model 2: Structural and Geographic Features**, extending the baseline Airbnb nightly price prediction model developed in the previous notebooks.

Model 1 used structural characteristics of Airbnb listings, such as accommodation type, capacity, bedrooms, beds, and bathrooms.

Model 2 introduces geographic information to investigate whether listing location improves predictive performance.

### Geographic Features

The following geographic variables will be considered:

- **Latitude and longitude:** continuous geographic coordinates.
- **Neighbourhood:** categorical information describing the listing's location within Milan.
- **Distance from the Duomo:** geographic distance from the city centre, calculated using the Haversine formula.

### Methodological Consistency

To ensure a meaningful comparison with Model 1, this notebook will preserve:

- The same listing selection criteria and target variable.
- The same structural predictors and preprocessing logic.
- The same 80/20 train/test split, stratified by room type.
- The same random state (`42`).

Preprocessing steps that learn from the data will be fitted only on the training data, preventing data leakage.

### Notebook Structure

1. Data Loading and Filtering
2. Feature Selection
3. Geographic Feature Engineering
4. Train/Test Split
5. Preprocessing Strategy
6. Data Validation
7. Final Summary

The resulting dataset and preprocessing strategy will be used in `06_geographic_model_training.ipynb` to train and evaluate Model 2.

# 2. Data Loading and Filtering

In [1]:
import pandas as pd
import numpy as np

from scipy import sparse
from sklearn.model_selection import train_test_split

# Load the original Inside Airbnb Milan dataset
df = pd.read_csv("../data/raw/listings.csv.gz")

print("Dataset shape:", df.shape)

Dataset shape: (25338, 90)


In [2]:
# Convert price strings to numeric values
df["price_numeric"] = pd.to_numeric(
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False),
    errors="coerce"
)

# Keep the same room types and price range as Model 1
selected_types = ["Entire home/apt", "Private room"]
MAX_PRICE = 1000

model_df = df.loc[
    df["room_type"].isin(selected_types)
    & df["price_numeric"].notna()
    & (df["price_numeric"] > 0)
    & (df["price_numeric"] <= MAX_PRICE)
].copy()

print("Listings after filtering:", len(model_df))
print("\nListings by room type:")
print(model_df["room_type"].value_counts())

assert len(model_df) == 21314, "Sample size differs from Model 1."

Listings after filtering: 21314

Listings by room type:
room_type
Entire home/apt    18877
Private room        2437
Name: count, dtype: int64


In [3]:
def add_bathroom_features(data):
    """
    Extract bathroom count and privacy type
    from the bathrooms_text column.
    """
    result = data.copy()

    text = result["bathrooms_text"].astype("string")

    numbers = pd.to_numeric(
        text.str.extract(
            r"(\d+(?:\.\d+)?)",
            expand=False
        ),
        errors="coerce"
    )

    half_bath = (
        text.str.contains("half-bath", case=False, na=False)
        & numbers.isna()
    )

    result["bathrooms_count"] = numbers.mask(half_bath, 0.5)

    text_lower = text.str.lower()

    result["bathroom_type"] = "unspecified"

    result.loc[
        text_lower.str.contains("private", na=False),
        "bathroom_type"
    ] = "private"

    result.loc[
        text_lower.str.contains("shared", na=False),
        "bathroom_type"
    ] = "shared"

    result.loc[
        text_lower.isna(),
        "bathroom_type"
    ] = "missing"

    return result


model_df = add_bathroom_features(model_df)

print("Missing bathroom counts:", model_df["bathrooms_count"].isna().sum())
print("\nBathroom types:")
print(model_df["bathroom_type"].value_counts())

Missing bathroom counts: 3

Bathroom types:
bathroom_type
unspecified    19137
private         1266
shared           908
missing            3
Name: count, dtype: int64


# 3. Feature Selection

In [4]:
# Structural features inherited from Model 1
structural_features = [
    "room_type",
    "property_type",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms_count",
    "bathroom_type"
]

# Geographic features to be added in Model 2
geographic_features = [
    "latitude",
    "longitude",
    "neighbourhood_cleansed"
]

target = "price_numeric"

# Check that all required columns are available
required_columns = structural_features + geographic_features + [target]

missing_columns = [
    col for col in required_columns
    if col not in model_df.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

print("Structural features:", len(structural_features))
print("Geographic features:", len(geographic_features))
print("All required columns are available.")

Structural features: 7
Geographic features: 3
All required columns are available.


# 4. Geographic Feature Engineering

In [5]:
# Duomo di Milano coordinates
DUOMO_LAT = 45.4642
DUOMO_LON = 9.1916

def haversine_distance(lat, lon, ref_lat, ref_lon):
    """
    Calculate the great-circle distance in kilometers
    between geographic coordinates and a reference point.
    """
    earth_radius_km = 6371.0

    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    ref_lat_rad = np.radians(ref_lat)
    ref_lon_rad = np.radians(ref_lon)

    delta_lat = lat_rad - ref_lat_rad
    delta_lon = lon_rad - ref_lon_rad

    a = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat_rad)
        * np.cos(ref_lat_rad)
        * np.sin(delta_lon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return earth_radius_km * c


# Create the geographic distance feature
model_df["distance_duomo_km"] = haversine_distance(
    model_df["latitude"],
    model_df["longitude"],
    DUOMO_LAT,
    DUOMO_LON
)

# Add the new feature to the geographic feature list
geographic_features.append("distance_duomo_km")

print("Geographic features:", geographic_features)
print("\nDistance from Duomo (km):")
print(model_df["distance_duomo_km"].describe())

Geographic features: ['latitude', 'longitude', 'neighbourhood_cleansed', 'distance_duomo_km']

Distance from Duomo (km):
count    21314.000000
mean         2.938461
std          1.555574
min          0.034394
25%          1.804624
50%          2.762989
75%          3.783369
max          9.844527
Name: distance_duomo_km, dtype: float64


In [6]:
# Define Model 2 predictors
model_features = structural_features + geographic_features

# Check missing values in predictors and target
missing_summary = (
    model_df[model_features + [target]]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing values by variable:")
print(missing_summary)

print("\nTotal listings:", len(model_df))
print("Total predictors:", len(model_features))

Missing values by variable:
beds                      432
bedrooms                    8
bathrooms_count             3
room_type                   0
property_type               0
accommodates                0
bathroom_type               0
latitude                    0
longitude                   0
neighbourhood_cleansed      0
distance_duomo_km           0
price_numeric               0
dtype: int64

Total listings: 21314
Total predictors: 11


# 5. Train/Test Split

In [7]:
# Define predictors and target
X = model_df[model_features].copy()
y = model_df[target].copy()

# Reproduce the Model 1 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=model_df["room_type"]
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining room types:")
print(X_train["room_type"].value_counts())

print("\nTest room types:")
print(X_test["room_type"].value_counts())

# Verify that training and test listings do not overlap
assert X_train.index.intersection(X_test.index).empty

# Verify that no listings were lost
assert len(X_train) + len(X_test) == len(model_df)

Training set: (17051, 11)
Test set: (4263, 11)

Training room types:
room_type
Entire home/apt    15101
Private room        1950
Name: count, dtype: int64

Test room types:
room_type
Entire home/apt    3776
Private room        487
Name: count, dtype: int64


# 6. Preprocessing Strategy

In [8]:
from sklearn.base import BaseEstimator, TransformerMixin

class BedsImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.medians_ = X.groupby("room_type")["beds"].median()
        self.global_median_ = X["beds"].median()
        return self

    def transform(self, X):
        X = X.copy()

        X["beds_missing"] = X["beds"].isna().astype(int)

        group_medians = X["room_type"].map(self.medians_)

        X["beds"] = (
            X["beds"]
            .fillna(group_medians)
            .fillna(self.global_median_)
        )

        return X

In [9]:
class RareCategoryGrouper(BaseEstimator, TransformerMixin):

    def __init__(self, column="property_type", min_frequency=30):
        self.column = column
        self.min_frequency = min_frequency

    def fit(self, X, y=None):
        counts = X[self.column].value_counts()

        self.frequent_categories_ = set(
            counts[counts >= self.min_frequency].index
        )

        return self

    def transform(self, X):
        X = X.copy()

        X[self.column] = (
            X[self.column]
            .where(
                X[self.column].isin(self.frequent_categories_),
                "Other"
            )
        )

        return X

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Numeric features: structural + geographic
numeric_features = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms_count",
    "beds_missing",
    "latitude",
    "longitude",
    "distance_duomo_km"
]

# Categorical features: structural + geographic
categorical_features = [
    "room_type",
    "property_type",
    "bathroom_type",
    "neighbourhood_cleansed"
]

# Preserve Model 1 preprocessing strategies
numeric_transformer = SimpleImputer(
    strategy="median"
)

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

# Combine numeric and categorical transformations
column_transformer = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 8
Categorical features: 4


In [11]:
# Complete preprocessing pipeline for Model 2
preprocessing_pipeline = Pipeline(
    steps=[
        ("beds_imputer", BedsImputer()),
        ("rare_categories", RareCategoryGrouper(
            column="property_type",
            min_frequency=30
        )),
        ("column_transformer", column_transformer)
    ]
)

print("Preprocessing pipeline:")
print(preprocessing_pipeline.named_steps.keys())

Preprocessing pipeline:
dict_keys(['beds_imputer', 'rare_categories', 'column_transformer'])


# 7. Data Validation

In [12]:
# Fit preprocessing only on training data
X_train_transformed = preprocessing_pipeline.fit_transform(
    X_train,
    y_train
)

print("Original training shape:", X_train.shape)
print("Transformed training shape:", X_train_transformed.shape)


# Check finite values in both sparse and dense matrices
if sparse.issparse(X_train_transformed):
    values_to_check = X_train_transformed.data
else:
    values_to_check = np.asarray(X_train_transformed)

assert np.isfinite(values_to_check).all(), (
    "Transformed training data contains NaN or infinite values."
)

print("All transformed values are finite.")

Original training shape: (17051, 11)
Transformed training shape: (17051, 116)
All transformed values are finite.


In [13]:
# Retrieve feature names after preprocessing
feature_names = (
    preprocessing_pipeline
    .named_steps["column_transformer"]
    .get_feature_names_out()
)

# Identify geographic features in the transformed dataset
geographic_output_features = [
    name for name in feature_names
    if (
        "latitude" in name
        or "longitude" in name
        or "distance_duomo_km" in name
        or "neighbourhood_cleansed" in name
    )
]

print("Total transformed features:", len(feature_names))
print("Geographic output features:", len(geographic_output_features))

print("\nFirst 15 geographic features:")
for name in geographic_output_features[:15]:
    print(name)

Total transformed features: 116
Geographic output features: 90

First 15 geographic features:
numeric__latitude
numeric__longitude
numeric__distance_duomo_km
categorical__neighbourhood_cleansed_ADRIANO
categorical__neighbourhood_cleansed_AFFORI
categorical__neighbourhood_cleansed_BAGGIO
categorical__neighbourhood_cleansed_BANDE NERE
categorical__neighbourhood_cleansed_BARONA
categorical__neighbourhood_cleansed_BICOCCA
categorical__neighbourhood_cleansed_BOVISA
categorical__neighbourhood_cleansed_BOVISASCA
categorical__neighbourhood_cleansed_BRERA
categorical__neighbourhood_cleansed_BRUZZANO
categorical__neighbourhood_cleansed_BUENOS AIRES - VENEZIA
categorical__neighbourhood_cleansed_CANTALUPA


In [14]:
# Apply the fitted preprocessing pipeline to the test set
X_test_transformed = preprocessing_pipeline.transform(X_test)

print("Training shape:", X_train_transformed.shape)
print("Test shape:", X_test_transformed.shape)

# Verify that both sets have the same number of features
assert X_train_transformed.shape[1] == X_test_transformed.shape[1]

# Check for invalid values in the transformed test set
if sparse.issparse(X_test_transformed):
    test_values = X_test_transformed.data
else:
    test_values = np.asarray(X_test_transformed)

assert np.isfinite(test_values).all(), (
    "Transformed test data contains NaN or infinite values."
)

print("Training and test preprocessing are consistent.")

Training shape: (17051, 116)
Test shape: (4263, 116)
Training and test preprocessing are consistent.


In [15]:
# Verify the presence of essential transformed features
essential_features = [
    "numeric__beds_missing",
    "numeric__latitude",
    "numeric__longitude",
    "numeric__distance_duomo_km"
]

missing_essential_features = [
    feature for feature in essential_features
    if feature not in feature_names
]

assert not missing_essential_features, (
    f"Missing essential features: {missing_essential_features}"
)

# Verify the number of encoded neighbourhood categories
neighbourhood_features = [
    feature for feature in feature_names
    if feature.startswith("categorical__neighbourhood_cleansed_")
]

print("Essential features verified:", len(essential_features))
print("Encoded neighbourhoods:", len(neighbourhood_features))
print("Total transformed features:", len(feature_names))
print("Final feature validation passed.")

Essential features verified: 4
Encoded neighbourhoods: 87
Total transformed features: 116
Final feature validation passed.


## 8. Final Summary

### Dataset Preparation

The dataset was prepared for **Model 2: Structural and Geographic Features**, extending the structural baseline developed in Model 1.

The same filtering criteria were preserved, resulting in **21,314 Airbnb listings**:

- 18,877 entire homes/apartments.
- 2,437 private rooms.

The target variable remains `price_numeric`.

### Geographic Feature Engineering

Four geographic predictors were introduced:

- `latitude`
- `longitude`
- `neighbourhood_cleansed`
- `distance_duomo_km`

Distance from Milan's Duomo was calculated using the Haversine formula.

No missing values were found in the geographic predictors, allowing all selected listings to be retained.

### Train/Test Split

The same splitting strategy as Model 1 was reproduced:

- Training set: 17,051 listings (80%).
- Test set: 4,263 listings (20%).
- Stratification by `room_type`.
- `random_state=42`.

The same filtering criteria, original row order, stratification variable, and random state used in Model 1 were preserved. Therefore, the train/test split is expected to contain the same listings as the baseline model, although listing-level membership was not independently verified.

### Preprocessing Pipeline

The preprocessing strategy from Model 1 was extended to incorporate geographic information.

The pipeline includes:

1. `BedsImputer`: imputes missing bed counts using room-type medians and creates a missing-value indicator.
2. `RareCategoryGrouper`: groups infrequent property types into an `"Other"` category.
3. `ColumnTransformer`: applies median imputation to numeric predictors and One-Hot Encoding to categorical predictors.

The neighbourhood variable was encoded without grouping rare neighbourhoods.

### Validation Results

The preprocessing pipeline was fitted on the training set and successfully applied to the test set.

| Validation metric | Result |
|---|---:|
| Original predictors | 11 |
| Transformed predictors | 116 |
| Encoded neighbourhoods | 87 |
| Training observations | 17,051 |
| Test observations | 4,263 |
| Missing or infinite transformed values | 0 |

The resulting feature matrices have consistent dimensions, and all essential structural and geographic predictors are retained.

### Next Steps

The next notebook, `06_geographic_model_training.ipynb`, will:

- Train regression models using structural and geographic features.
- Compare alternative representations of geographic information.
- Use cross-validation on the training set for model selection.
- Evaluate the selected model on the test set.
- Compare predictive performance against the structural baseline (Model 1).

**The objective is to quantify the additional predictive value of geographic information while preserving methodological consistency with Model 1.**